# Generic EDA Skeleton — and What Changes If You're Blind

Same dataset as `01_eda.ipynb` (`mlg-ulb/creditcardfraud`), but this notebook is organized
strictly around a domain-agnostic sequence:

**quality check → target understanding → signal-finding → naive baseline**

At the last two phases, we run two parallel tracks:

- **Labeled path** — what we did in `01_eda.ipynb`, using `Class` to guide the analysis.
- **Blind path** — what the same phase looks like if `Class` doesn't exist, simulated by
  literally dropping the column before doing signal-finding and baseline work. We only look
  at `Class` again at the very end, to see what the blind path actually got us — a real blind
  analysis wouldn't have that luxury.

In [1]:
import pandas as pd

from fraud_pipeline.ingestion.loader import load_transactions

df = load_transactions()
df.shape

(284807, 31)

## Phase 1 — Quality check

Shape, types, missingness, duplicates. This phase never depends on having a target — it's
identical whether you're blind or not, which is why it's always step one regardless of the
problem.

In [2]:
print("shape:", df.shape)
print()
print("dtypes:")
print(df.dtypes.value_counts())
print()
print("missing values (total):", df.isna().sum().sum())
print("duplicate rows:", df.duplicated().sum())

shape: (284807, 31)

dtypes:
float64    30
int64       1
Name: count, dtype: int64

missing values (total): 0
duplicate rows: 1081


## Phase 2 — Target understanding

**Labeled path:** we know `Class` is the target, so we characterize it directly — this is
where the 0.17% fraud rate comes from, and it's the single most important number in the
whole project (it's why accuracy is a useless metric here, and why the naive baseline in
Phase 4 has to be measured on recall/false-positive rate instead).

In [3]:
df["Class"].value_counts(normalize=True)

Class
0    0.998273
1    0.001727
Name: proportion, dtype: float64

**Blind path:** there is no target to characterize — in a real blind scenario you likely
wouldn't even know a `Class`-like concept exists yet, only that you have 30 numeric columns.
This phase effectively collapses: the closest analog is the overall column spread already
visible in `describe()` from Phase 1. Nothing new happens here without a target to anchor
on — which is itself informative. It means "target understanding" isn't a phase you can
force; it's a phase that either exists or doesn't, depending on whether labels exist.

## Phase 3 — Signal-finding

**Labeled path:** rank features by how well they separate the target (Cohen's d against
`Class`), exactly as in `01_eda.ipynb`.

In [4]:
feature_cols = [c for c in df.columns if c.startswith("V")]
fraud = df[df["Class"] == 1]
normal = df[df["Class"] == 0]


def cohens_d(a, b):
    pooled_std = ((a.std() ** 2 + b.std() ** 2) / 2) ** 0.5
    return (a.mean() - b.mean()) / pooled_std


separation = pd.Series(
    {col: cohens_d(fraud[col], normal[col]) for col in feature_cols}
).sort_values(key=abs, ascending=False)

labeled_top6 = separation.abs().sort_values(ascending=False).index[:6].tolist()
print("Top 6 by separation from target:", labeled_top6)

Top 6 by separation from target: ['V14', 'V4', 'V11', 'V12', 'V10', 'V16']


**Blind path:** drop `Class` entirely. With no target to rank against, "signal-finding"
has to fall back on some property of the features themselves — the standard unsupervised
proxy is variance: features that vary more are assumed to carry more information. Note this
is a real assumption, not a fact — it's about to be tested.

In [5]:
df_blind = df.drop(columns=["Class"])

variance_ranking = df_blind[feature_cols].std().sort_values(ascending=False)
blind_top6 = variance_ranking.index[:6].tolist()
print("Top 6 by raw variance (blind):", blind_top6)

Top 6 by raw variance (blind): ['V1', 'V2', 'V3', 'V4', 'V5', 'V6']


Compare the two lists. They will barely overlap — and there's a structural reason why:
these `V` columns are PCA components, and PCA orders its components by variance *by
construction*. Ranking by variance on PCA output just recovers `V1, V2, V3, ...` in roughly
their original order — it tells you nothing about fraud, only about how the anonymization
was computed. The blind path's "signal-finding" step isn't actually finding signal here;
it's rediscovering an artifact of the upstream transform.

## Phase 4 — Naive baseline

Same rule both times (Tukey's IQR: flag anything past 1.5×IQR from Q1/Q3), applied to each
path's own top-6 features. This isolates the effect of *which features got picked* in Phase
3 — the detection logic itself is held constant.

In [6]:
def iqr_outlier_mask(series, k=1.5):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    return (series < q1 - k * iqr) | (series > q3 + k * iqr)


def evaluate_rule(feature_list, label="path"):
    flags = pd.DataFrame({col: iqr_outlier_mask(df[col]) for col in feature_list})
    flagged = flags.any(axis=1)
    recall = flagged[df["Class"] == 1].mean()
    fpr = flagged[df["Class"] == 0].mean()
    flag_rate = flagged.mean()
    print(f"[{label}] features: {feature_list}")
    print(f"[{label}] flag rate (share of all transactions flagged): {flag_rate:.1%}")
    print(f"[{label}] recall (fraud caught):                          {recall:.1%}")
    print(f"[{label}] false positive rate (normal flagged):           {fpr:.1%}")
    print()
    return flagged


labeled_flags = evaluate_rule(labeled_top6, "labeled")

[labeled] features: ['V14', 'V4', 'V11', 'V12', 'V10', 'V16']
[labeled] flag rate (share of all transactions flagged): 16.1%
[labeled] recall (fraud caught):                          91.7%
[labeled] false positive rate (normal flagged):           16.0%



The labeled path can report all three numbers because it has `Class` the whole way through.
The blind path, run honestly, could only ever report the flag rate — recall and false
positive rate are computed here purely for comparison, using information a genuinely blind
pipeline would not have access to.

In [7]:
blind_flags = evaluate_rule(blind_top6, "blind (peeking at Class for comparison only)")

[blind (peeking at Class for comparison only)] features: ['V1', 'V2', 'V3', 'V4', 'V5', 'V6']
[blind (peeking at Class for comparison only)] flag rate (share of all transactions flagged): 15.6%
[blind (peeking at Class for comparison only)] recall (fraud caught):                          78.7%
[blind (peeking at Class for comparison only)] false positive rate (normal flagged):           15.5%

